# Automação dos relatórios de Colaboradores

- Extrair do sistema Datamace o relatório de movimentação de Headcount através do módulo GIP
- Tratamento da base e carga do COLAB
- Atualização de Power BI Headcount
- Atualização da planilha Controle_HC e Atestados

# Importando as Bibliotecas

In [ ]:
import duckdb as db
import os
import pandas as pd
from pathlib import Path
import pyautogui
import pyperclip
import shutil
import time
import tkinter as tk
import win32com.client
from datetime import datetime, date
from openpyxl import load_workbook, Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.worksheet.table import Table, TableStyleInfo
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver import ActionChains
from tkinter import simpledialog
from webdriver_manager.chrome import ChromeDriverManager

tempo_0 = [id, datetime.today(), 0]

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Processo de Importação finalizado')
print('')
print('----------------------------------------------------------------------------------------------------')

# Caminhos de Pastas

In [ ]:
CONFIG = {
    'id_processo': 7,
    'caminho_registros': Path(r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PROCESSOS.xlsx'),
    'diretorio_extracao': Path(r'C:\Users\rodrigo.bernandes\Downloads'),
    'diretorio_movidos': Path(r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\2. ARQUIVOS MOVIDOS'),
    'arquivo_final': Path(r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\COLABORADORES.xlsx'),
}

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Caminhos de pastas definidas')
print('')
print('----------------------------------------------------------------------------------------------------')

# Registra etapa do processo

In [ ]:
def append_registro_processo(caminho, id_proc, etapa):
    try:
        wb = load_workbook(caminho)
        ws = wb['REGISTROS']
        ws.append([id_proc, datetime.today(), etapa])
        wb.save(caminho)
        wb.close()
    except Exception as e:
        print(f"❌ Erro ao registrar etapa {etapa}: {e}")

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Registro de processos')
print('')
print('----------------------------------------------------------------------------------------------------')

# Configurações Iniciais

In [ ]:
automacao = r'X:\Gestao_de_Pessoas\Analytics\08 - Notebooks Python\08.3 - Automações\AUTOMACAO'

chrome_options = Options()
chrome_options.add_argument(f"--user-data-dir={automacao}")
chrome_options.add_argument("--profile-directory=Default")
chrome_options.add_argument("--remote-allow-origins=*")
chrome_options.add_argument("--start-maximized")
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

try:
    driver.get("https://app.dtm.com.br/")
    wait = WebDriverWait(driver, 150)
    actions = ActionChains(driver)
    
    print("⏳ Sincronizando Datamace...")

except Exception as e:
    print(f"❌ Erro Geral: {e}")

finally:
    print(f"🏁 Acesso finalizado")

# Acessando o Datamace

In [ ]:
pyautogui.PAUSE = 1

# Área de login na Cloud Datamace
pyautogui.click(x=748, y=381, clicks=2)
pyautogui.write("afpesp-17")
pyautogui.press("tab")
pyautogui.write("5SSWi3VqS")
pyautogui.press("tab")
pyautogui.press("enter")
time.sleep(30) # Tempo maior para aguardar carregar a página de login

# Área de login do usuário
pyautogui.click(x=836, y=445)
pyautogui.write("RODRIGO")
pyautogui.press("tab")
pyautogui.write("Roddydio/")
pyautogui.press("tab")
pyautogui.press("enter")

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Acesso ao Datamace realizado com sucesso')
print('')
print('----------------------------------------------------------------------------------------------------')

# Dentro do Datamace, acessando o GIP

In [ ]:
# Acessando o GIP
time.sleep(10) # Tempo maior para aguardar carregar o GIP
pyautogui.click(x=168, y=185) # Maximiza a janela
time.sleep(2)
pyautogui.click(x=168, y=185) # Clica no GIP
time.sleep(5)
pyautogui.click(x=1079, y=240) # Fecha janela de Tarefas Anuais 
time.sleep(1)
pyautogui.click(x=1109, y=301) # Geradores
time.sleep(1)
pyautogui.click(x=1168, y=325) # Arq./Relatório 
time.sleep(1)
pyautogui.click(x=546, y=364) # Lupa
time.sleep(1)
pyautogui.press("enter")
time.sleep(1)
pyautogui.click(x=615, y=614) # Arquivo
time.sleep(1)
pyautogui.click(x=685, y=681) # Confirmar

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Extraindo relatório')
print('')
print('----------------------------------------------------------------------------------------------------')

# Aguardando a Conclusão do Download da base COLAB

In [ ]:
root = tk.Tk()
root.withdraw()  # não mostra janela principal

while True:
    tecla = simpledialog.askstring("Continuar", "Digite S para continuar:")
    if (tecla or "").strip().upper() == "S":
        break

print(" ✅ Continuando o processo...")

# Retorna arquivos COLAB presentes no diretório

In [ ]:
def obter_arquivos_colaboradores(diretorio):
    if not diretorio.exists():
        print(f"❌ Diretório não existe: {diretorio}")
        return []
    
    lista_arquivos = [f.name for f in diretorio.glob('COLAB*.xlsx')]
    return lista_arquivos

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Arquivo COLAB na pasta')
print('')
print('----------------------------------------------------------------------------------------------------')

# Processa arquivo de colaboradores com DuckDB

In [ ]:
def processar_colaboradores(arquivo):
    df = pd.read_excel(arquivo)
    
    df_processado = db.query("""
        SELECT      
            REGISTRO AS registro,
            NOME_COMP AS nome,
            NASCTO AS nascimento,
            SEXO AS sexo,
            RACA_COR AS etnia_raca,
            FORMACAO AS cod_formacao,
            APOSENTADO AS aposentado,
            RG_NUMERO AS rg, 
            CPF_NUMERO AS cpf,
            DEFICIENTE AS deficiente,
            NACIONALI AS nacionalidade,
            "NAT.UF_ORI" AS naturalidade_uf,
            NAT_CIDADE AS naturalidade_cidade,
            EST_CIVIL AS estado_civil,
            SEXO_CONJU AS sexo_conjuge,
            NOME_CONJU AS nome_conjuge,
            DT_NAS_CON AS data_nasc_conjuge,
            CPF_CONJUG AS cpf_conjuge,
            FILHOS AS filhos,
            ADM_DATA AS data_admissao,
            SITU_DESCR AS situacao,
            RESC_DAT AS data_rescisao,
            RESC_DESCR AS descricao_rescisao,
            COD_EMPRES AS cod_empresa,            
            TAB_CARGO AS cod_cargo,
            CARGO_COMP AS cargo_completo,
            CARGO_DRES AS cargo_abreviado,
            TAB_CC_CON AS cod_centro_custo,
            NOM_GEST_C AS nome_gestor,
            TAB_SECAO AS cod_secao,
            SECAO AS secao,
            CCUSTO_CON AS centro_custo,
            SAL_ADMISS AS salario_admissao,
            SAL_ATUAL AS salario_atual,
            SAL_TOTAL AS salario_total,
            FGT_SALDO AS saldo_fgts,
            ENDERECO AS endereco,
            END_NUMERO AS numero_endereco,
            COMPLEMEN AS complemento,
            BAIRRO AS bairro,
            CIDADE AS cidade,
            ESTADO AS estado,
            TELEFONE1 AS telefone,
            E_MAIL AS email,
            SITUACAO AS cod_situacao,
            REG_TRABAL AS regime_trabalho,
            HOR_BASE AS hora_base,
            "HOR_COMPL." AS hora_complemento,
            PRO_DT_IND AS ultimo_reajuste_individual,
            PRO_DT_COL AS ultimo_reajuste_coletivo,
            FER_DT_INI AS data_inicio_ferias,
            FER_DT_FIM AS data_fim_ferias
        FROM (
            SELECT      
                *,
                ROW_NUMBER() OVER(  
                    PARTITION BY REGISTRO
                    ORDER BY SITUACAO ASC,
                    CASE WHEN RESC_DAT IS NULL THEN '2050-12-31' ELSE RESC_DAT END DESC
                ) AS ordem
            FROM df
        ) AS sub_query
        WHERE ordem = 1
    """).df()
    
    return df_processado

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Tratamento da base COLAB finalizado')
print('')
print('----------------------------------------------------------------------------------------------------')

# Cria arquivo Excel com formatação de tabela

In [ ]:
def criar_arquivo_excel(df, caminho, nome_tabela="COLABORADORES"):

    wb = Workbook()
    ws = wb.active
    ws.title = nome_tabela
    
    for r in dataframe_to_rows(df, index=False, header=True):
        ws.append(r)
    
    tabela = Table(displayName=nome_tabela, ref=ws.dimensions)
    estilo = TableStyleInfo(
        name="TableStyleLight13",
        showFirstColumn=False,
        showLastColumn=False,
        showRowStripes=True,
        showColumnStripes=True
    )
    tabela.tableStyleInfo = estilo
    ws.add_table(tabela)
    
    wb.save(caminho)
    wb.close()

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Arquivo COLABORADORES criado')
print('')
print('----------------------------------------------------------------------------------------------------')

# Finalizando processo

In [ ]:
# Execução Principal
def main():
    config = CONFIG
    inicio = datetime.now()
    
    try:
        # Etapa 0
        append_registro_processo(config['caminho_registros'], config['id_processo'], 0)
        
        # Etapa 1 - Listar arquivos
        arquivos = obter_arquivos_colaboradores(config['diretorio_extracao'])
        
        if not arquivos:
            print("⚠️ Nenhum arquivo COLAB encontrado")
            return
        
        append_registro_processo(config['caminho_registros'], config['id_processo'], 1)
        
        # Processar arquivos
        for arquivo_nome in arquivos:
            try:
                arquivo_caminho = config['diretorio_extracao'] / arquivo_nome
                print(f"🔄 Processando: {arquivo_nome}")
                
                df_processado = processar_colaboradores(arquivo_caminho)
                criar_arquivo_excel(df_processado, config['arquivo_final'])
                
                # Mover arquivo
                destino = config['diretorio_movidos'] / arquivo_nome
                shutil.move(str(arquivo_caminho), str(destino))
                
                print(f"✅ {arquivo_nome} concluído")
                
            except Exception as e:
                print(f"❌ Erro ao processar {arquivo_nome}: {e}")
                continue
        
        # Etapa 8 - Finalização
        append_registro_processo(config['caminho_registros'], config['id_processo'], 8)
                
        duracao = datetime.now() - inicio
        
        print('—' * 100)
        print(f'\n   ✅ Processo finalizado com sucesso')
        print(f'   ⏱️  Tempo de execução: {duracao}\n')
        print('—' * 100)
        
    except Exception as e:
        print(f"❌ Erro crítico na execução: {e}")
        raise

if __name__ == "__main__":
    main()

# Atualização do Power BI - HEADCOUNT

In [ ]:
pyautogui.PAUSE = 1

# Entrar no Power BI
path_pbi = r"X:\Gestao_de_Pessoas\Analytics\09 - Power BI\HEADCOUNT.pbix"
os.startfile(path_pbi) # Abre o arquivo pelo Windows
time.sleep(40)

# Atualizar Power BI
pyautogui.click(x=753, y=103) # Clicar em Atualizar
time.sleep(40)
pyautogui.click(x=1294, y=109) # Publicar
time.sleep(5)
pyautogui.click(x=863, y=477) # Salvar
time.sleep(5)
pyautogui.click(x=712, y=379) # Clicar em GESTÃO DE PESSOAS
pyautogui.press("tab")
pyautogui.press("enter")
time.sleep(7)
pyautogui.click(x=871, y=577) # Substituir
time.sleep(10)
pyautogui.click(x=990, y=585) # Clicar em Entendi
time.sleep(5)
pyautogui.hotkey("Alt", "F4")
time.sleep(2)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Power BI Atualizado')
print('')
print('----------------------------------------------------------------------------------------------------')

# Atualizando o arquivo Excel Controle HC e Atestados
- Caminho: X:\Gestao_de_Pessoas\Analytics\10 - Relatórios\10.4 - HC e Atestados Médicos

In [ ]:
# Caminho do arquivo
path_excel = r"X:\Gestao_de_Pessoas\Analytics\10 - Relatórios\10.4 - HC e Atestados Médicos\Controle_HC e Atestados.xlsb"
os.startfile(path_excel) # Abre o arquivo pelo Windows
time.sleep(60)
pyautogui.press('esc')
time.sleep(15)

# Utiliza o comando "Ir para" (Ctrl + G) para navegar até a aba e célula
pyautogui.hotkey('ctrl', 'g')
time.sleep(1)

# Digita o endereço completo
pyautogui.write('HC!B5')
time.sleep(1)
pyautogui.press('enter')
time.sleep(3)

#Atualizando o arquivo
pyautogui.hotkey('alt', 'F5')
time.sleep(2)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Planilha atualizada com sucesso')
print('')
print('----------------------------------------------------------------------------------------------------')

# Resumo de Finalização do Processo

In [ ]:
tempo_1 = [id, datetime.today(), 8]

print('----------------------------------------------------------------------------------------------------')
print('')
print('     ✅  Processo finalizado')
print('')
print('     ⏱️   Tempo de execução:')
print('')
print(f'   {tempo_1[1] - tempo_0[1]}')
print('')
print('----------------------------------------------------------------------------------------------------')